In [2]:
# Problem Statement
# Identify charges for insurance based on the metrics provided.
# After looking into the dataset its numerical data. 
# Both input and outputs are available.
# Output label is numerical.
###################################################
# Domain - MACHINE LEARNING
# Supervised Learning
# Regression
###################################################

In [15]:
import pandas as pd

In [16]:
dataset = pd.read_csv("insurance_pre.csv")
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
# Dataset has 1338 Rows X 6 Columns
# We have to convert the following nominal fields to numerical using one-hot encoding.
# The nominal fields are smoker, sex.

In [17]:
# Preprocessing
dataset = pd.get_dummies(dataset, dtype="int64", drop_first=True)
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [18]:
#  Display Columns
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [19]:
# Identify independent variables
independent = dataset[["age", "bmi", "children", "sex_male", "smoker_yes"]]
independent

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [20]:
# Identify dependent variable
dependent = dataset[["charges"]]
dependent

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [21]:
# Split training and test set
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(independent, dependent, test_size = 0.30, random_state = 0)

In [22]:
# --------------------------------------------------
# Feature Scaling (Required for SVR)
### StandardScaler — Quick Notes
# * **StandardScaler** standardizes features so they have approximately **mean = 0** and **standard deviation = 1**.
# * Formula: **`z = (x - mean) / standard deviation`**
# * `fit()` → learns the **mean and standard deviation** from the training data.
# * `transform()` → applies the learned scaling to the data.
# * `fit_transform()` → performs both `fit()` and `transform()` together.
# * Use `fit_transform()` on **training data**.
# * Use only `transform()` on **test data** to avoid **data leakage**.
# * Scaling is especially important for **SVR, KNN, SVM, PCA, and neural networks**.
# * Scaling is generally **not required for Decision Trees, Random Forest, AdaBoost, XGBoost, and LightGBM**.
# --------------------------------------------------
from sklearn.preprocessing import StandardScaler
# Create the scaler
scaler = StandardScaler()
# Fit on training data and transform
x_train = scaler.fit_transform(x_train)
# Transform the test data using the same scaler
x_test = scaler.transform(x_test)

In [23]:
# Import GridSearchCV, which automatically tries different hyperparameter combinations
# and finds the combination that gives the best cross-validation score
from sklearn.model_selection import GridSearchCV

# Import Support Vector Regression algorithm
from sklearn.svm import SVR

# Define the hyperparameters and the values that we want GridSearchCV to try
param_grid = {
    # Try four different types of kernels
    'kernel': ['rbf', 'poly', 'sigmoid', 'linear'],

    # Try different values of C, which controls the trade-off between model complexity
    # and allowing errors
    'C': [10, 100, 1000, 2000, 3000],

    # Try both automatic and scaled calculations for gamma
    'gamma': ['auto', 'scale']
}

# Create the GridSearchCV object
# SVR() is the model we want to tune
# param_grid contains all the hyperparameter combinations to test
# refit=True means the best model will be trained again using the complete training data
# verbose=3 displays detailed information about the progress
# n_jobs=-1 uses all available CPU cores to speed up the search
grid = GridSearchCV(
    SVR(),
    param_grid,
    refit=True,
    verbose=3,
    n_jobs=-1
)

# Run the hyperparameter search on the training data
# GridSearchCV trains and evaluates SVR for every combination in param_grid
grid.fit(x_train, y_train)
print("completed")

Fitting 5 folds for each of 40 candidates, totalling 200 fits
completed


C:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [25]:
# Print best parameter after tuning
re = grid.cv_results_
table = pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.017199,0.002054,0.008572,0.000491,10,auto,rbf,"{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}",0.004055,0.013366,-0.103821,-0.095119,-0.101604,-0.056625,0.053504,35
1,0.014668,0.003309,0.002637,0.000495,10,auto,poly,"{'C': 10, 'gamma': 'auto', 'kernel': 'poly'}",0.056274,0.069532,-0.045601,-0.025079,-0.049592,0.001107,0.051309,32
2,0.029365,0.001153,0.005524,0.000924,10,auto,sigmoid,"{'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}",0.049905,0.075905,-0.046585,-0.041004,-0.046507,-0.001657,0.053391,34
3,0.016537,0.003285,0.003615,0.000864,10,auto,linear,"{'C': 10, 'gamma': 'auto', 'kernel': 'linear'}",0.377969,0.479601,0.317872,0.337979,0.324422,0.367569,0.059777,25
4,0.019284,0.001770,0.009546,0.000922,10,scale,rbf,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.004126,0.013244,-0.103775,-0.095165,-0.101602,-0.056634,0.053486,36
5,0.014504,0.003109,0.002968,0.001039,10,scale,poly,"{'C': 10, 'gamma': 'scale', 'kernel': 'poly'}",0.054964,0.071297,-0.046513,-0.024157,-0.049652,0.001188,0.051594,31
6,0.025172,0.002091,0.005705,0.000866,10,scale,sigmoid,"{'C': 10, 'gamma': 'scale', 'kernel': 'sigmoid'}",0.049644,0.076323,-0.046798,-0.040824,-0.046521,-0.001635,0.053474,33
7,0.017339,0.001566,0.003689,0.000496,10,scale,linear,"{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}",0.377969,0.479601,0.317872,0.337979,0.324422,0.367569,0.059777,25
8,0.020677,0.001659,0.009301,0.001146,100,auto,rbf,"{'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}",0.300573,0.339474,0.173708,0.217991,0.183375,0.243024,0.065733,29
9,0.016494,0.002907,0.003026,0.000469,100,auto,poly,"{'C': 100, 'gamma': 'auto', 'kernel': 'poly'}",0.540643,0.575051,0.474839,0.535730,0.424640,0.510181,0.053582,21


In [28]:
from sklearn.metrics import r2_score
grid_predictions = grid.predict(x_test)
r_score = r2_score(y_test, grid_predictions)
print("The R2 value for best params {}: ".format(grid.best_params_), r_score)

The R2 value for best params {'C': 3000, 'gamma': 'scale', 'kernel': 'poly'}:  0.8598930084494356


In [30]:
# Print best parameter after tuning
re = grid.cv_results_
grid_predictions = grid.predict(x_test)
grid_predictions

array([ 9882.23553045,  8684.53726052, 47863.91492801, 12571.3296839 ,
       10503.41343965,  4754.93723665,  1443.94974332, 11161.657231  ,
        7435.84538836,  5371.62749203,  6792.36540753, 10061.14110577,
        7632.07502973,  4871.76871538, 25340.31666642, 10702.78951231,
       12523.59208885,  3559.80058082,  6538.68332237, 24014.94367286,
       25935.15514707, 12148.61708249,  9691.29174851, 30533.71413143,
        1963.47262983,  4967.98361807,  4445.33159778,  8003.49970394,
        4385.68529386,  8313.99557978,  7894.30531574, 51334.29195097,
       13004.91484206, 10251.92921715, 12711.42308371,  4153.88138193,
        8739.42694068, 35968.17012455, 35436.61473361,  2077.05999458,
        5945.32507383,  3376.59733561, 27298.90556466, 44665.58571445,
       32712.02939249,  3248.44366366, 10704.52721091,  7044.43520973,
        4729.54504139, 12142.93055349,  2793.08923534,  2960.73592137,
       30936.72609045, 42934.61243677, 11849.41436963,  3311.33971211,
      

In [32]:
age = float(input("Age: "))
bmi = float(input("BMI: "))
children = float(input("Children: "))
sex = float(input("Sex Male 0 or 1: "))
smoker = float(input("Smoker Yes 0 or 1: "))
future_prediction = grid.predict(
    [[age, bmi, children, sex, smoker]]
)
future_prediction

Age:  32
BMI:  7.5
Children:  1
Sex Male 0 or 1:  1
Smoker Yes 0 or 1:  0


array([6572971.48380077])